# 03 — Train Neural Collaborative Filtering (NCF) on Colab GPU

This notebook trains the NeuMF model (from `src/models/ncf.py`) on the full MovieLens 25M dataset using Colab's free T4 GPU. Training takes ~15–20 minutes.

**Architecture:** NeuMF = GMF branch (linear) + MLP branch (nonlinear) fused via sigmoid. See `src/models/ncf.py` and the paper: https://arxiv.org/abs/1708.05031

**Output:** `ncf.pth` checkpoint, downloaded locally and (optionally) uploaded to a GitHub Release for use by `make evaluate`.

## 1. Setup

In [ ]:
import os, sys, time, urllib.request, zipfile
import numpy as np, pandas as pd, torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'torch {torch.__version__}  device={device}')
assert device == 'cuda', 'Runtime -> Change runtime type -> T4 GPU'

## 2. Clone the repo (to get `src/models/ncf.py`)

We clone the project so we import the SAME model code that `make evaluate` uses locally. No copy-paste drift.

In [ ]:
if not os.path.isdir('recsys-engine'):
    !git clone https://github.com/its-Sohan/recsys-engine.git
sys.path.insert(0, 'recsys-engine')

from src.models.ncf import NeuMF, NCFDataset, NCFRecommender
print('NCF modules imported')

## 3. Download MovieLens 25M

Direct download in Colab (~250MB, ~1 min).

In [ ]:
DATA_DIR = 'ml-25m'
if not os.path.isdir(DATA_DIR):
    url = 'https://files.grouplens.org/datasets/movielens/ml-25m.zip'
    print('Downloading...')
    urllib.request.urlretrieve(url, 'ml-25m.zip')
    with zipfile.ZipFile('ml-25m.zip') as z:
        z.extractall('.')
    os.remove('ml-25m.zip')
print('Dataset ready:', os.listdir(DATA_DIR))

## 4. Load + time-based split

Same split logic as `src/data/loader.py` — test = latest 20% by timestamp. We only need train for NCF; eval happens locally.

In [ ]:
ratings = pd.read_csv(f'{DATA_DIR}/ratings.csv',
    dtype={'userId': np.int32, 'movieId': np.int32, 'rating': np.float32, 'timestamp': np.int64})
ratings = ratings.sort_values('timestamp').reset_index(drop=True)
n_test = int(len(ratings) * 0.2)
train = ratings.iloc[:-n_test].copy()
test  = ratings.iloc[-n_test:].copy()
print(f'train: {len(train):,}  test: {len(test):,}')

## 5. Build id maps + dataset with negative sampling

In [ ]:
users = np.sort(train.userId.unique())
items = np.sort(train.movieId.unique())
user2idx = {u: i for i, u in enumerate(users)}
item2idx = {it: i for i, it in enumerate(items)}
idx2item = {i: it for it, i in item2idx.items()}
print(f'{len(user2idx):,} users, {len(item2idx):,} items')

from torch.utils.data import DataLoader
dataset = NCFDataset(train, user2idx, item2idx, neg_ratio=4, n_items=len(item2idx))
loader = DataLoader(dataset, batch_size=1024, shuffle=True, num_workers=2)
print(f'training tuples: {len(dataset):,}  (positives + 4x negatives)')

## 6. Train NeuMF on GPU

~15-20 min for 15 epochs on T4. Loss should drop from 0.6931 (= ln2, random) to ~0.30-0.40.

In [ ]:
model = NeuMF(len(user2idx), len(item2idx)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.BCELoss()

EPOCHS = 15
model.train()
for epoch in range(EPOCHS):
    t0 = time.time()
    total, n = 0.0, 0
    for bu, bi, bl in loader:
        bu, bi, bl = bu.to(device), bi.to(device), bl.to(device)
        optimizer.zero_grad()
        preds = model(bu, bi)
        loss = criterion(preds, bl)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(bu)
        n += len(bu)
    print(f'epoch {epoch+1:2d}/{EPOCHS}  loss={total/n:.4f}  ({time.time()-t0:.0f}s)')

## 7. Save the `.pth` checkpoint

We save the full payload (state_dict + id maps + seen sets) so `NCFRecommender.load()` can reconstruct everything locally without re-training.

In [ ]:
seen = train.groupby('userId')['movieId'].apply(set).to_dict()
payload = {
    'state_dict': model.state_dict(),
    'config': {'gmf_dim': 64, 'mlp_embed_dim': 32, 'mlp_layers': [64, 32, 16, 8]},
    'user2idx': user2idx,
    'idx2item': {int(k): int(v) for k, v in idx2item.items()},
    'seen': {int(k): list(v) for k, v in seen.items()},
}
torch.save(payload, 'ncf.pth')
print(f'Saved ncf.pth ({os.path.getsize("ncf.pth")/1e6:.1f} MB)')
from google.colab import files
files.download('ncf.pth')

## 8. (Optional) Upload to a GitHub Release

If you'd rather not manage the file locally, upload it as a GitHub Release asset so the Dockerfile can fetch it at build time. Create a release on GitHub (tag: `ncf-v1`), then:

```
!pip install -q ghapi
# Then use the GitHub CLI or API to upload ncf.pth to the release.
# Or just upload via the GitHub web UI: Releases -> ncf-v1 -> Upload asset.
```

After uploading, place the file at `artifacts/ncf.pth` locally and run:
```
make evaluate --models ncf
```

## What just happened (read this before interviews)

1. **GMF branch** learned linear user-item interactions (generalized dot product).
2. **MLP branch** learned nonlinear interactions via stacked dense layers.
3. **NeuMF fusion** learned how to weight both branches via a final sigmoid.
4. **Negative sampling** converted explicit star ratings into implicit binary signal — for each real positive, 4 random negatives were generated.
5. **BCE loss** trained the model to output P(user interacts with item).

Loss should have dropped from ~0.693 (ln 2 = random guessing) to ~0.30-0.40. That drop is the model learning user tastes.